If you want to type along with me, head to [this notebook](https://humboldt.cloudbank.2i2c.cloud/hub/user-redirect/git-pull?repo=https%3A%2F%2Fgithub.com%2Fmlekimchi%2Fdata111_fall26&branch=main&urlpath=tree%2Fdata111_fall26%2Flectures%2Flec15_live.ipynb) instead. If you prefer follow along by executing the cells, stay in this notebook.

# Lecture 15: Chance and Sampling
v.ekc

Simulation *estimates* probabilities; today we also learn to *calculate* them exactly — then meet random sampling, where probability and data science officially merge. (Slides: KL Lecture 15 deck.)

**Today's sections:**
1. Chance — the three rules
2. Random sampling
3. Distributions: probability vs. empirical
4. Large random samples

In [ ]:
from datascience import *
import numpy as np

%matplotlib inline
import matplotlib.pyplot as plots
plots.style.use('fivethirtyeight')

---
## 1. Chance — The Three Rules 🎲

A box holds cards R, B, G; we draw two **without replacement**. For each question we'll simulate first, then calculate — they should agree!

### 📋 Board Reference

| Rule | Formula | Use when |
|---|---|---|
| Multiplication | P(A **and** then B) = P(A) · P(B given A) | both events happen |
| Addition | P(one of several ways) = sum of the ways | ways don't overlap |
| Complement | P(at least one) = 1 − P(none) | "at least one" questions |

Box contains cards labeled R, B, and G. I **draw two cards at random without replacement**.

### Chance of two events both occurring ###
What is the chance that I get G followed by B?

(both "get G first" and "get B second")

In [ ]:
cards = make_array('R','B','G')

In [ ]:
np.random.choice(cards, 2,replace=False)

In [ ]:
def green_then_blue():
    picks = np.random.choice(cards, 2,replace=False)
    return picks.item(0) =='G' and picks.item(1) =='B'

In [ ]:
outcomes = make_array()

for i in np.arange(10000):
    outcomes = np.append(outcomes,green_then_blue())
    
outcomes

In [ ]:
sum(outcomes)/10000

In [ ]:
np.average(outcomes)

In [ ]:
1/3*1/2 

> **Multiplication rule in action:** P(G first) = 1/3, then P(B second given G gone) = 1/2 → 1/3 · 1/2 = 1/6 ≈ what the simulation said. ✅

### Chance of an event that can happen in multiple ways ###
What is the chance that one of the ticket is G and the other is B?

In [ ]:
def green_and_blue():
    picks = np.random.choice(cards, 2,replace=False)
    return (picks.item(0) =='G' and picks.item(1) =='B') or (picks.item(0) =='B' and picks.item(1) =='G')

In [ ]:
outcomes = make_array()

for i in np.arange(10000):
    outcomes = np.append(outcomes,green_and_blue())
    
outcomes

In [ ]:
np.average(outcomes)

In [ ]:
1/3*1/2 + 1/3*1/2

> **Addition rule:** "G then B" and "B then G" are two non-overlapping ways to get {G, B} — so add them: 1/6 + 1/6 = 1/3.

### Chance of At Least One Success in Independently Repeated Success/Failure Trials ###

In [ ]:
# Chance of no sixes in 4 rolls of a die

prob_no_sixes_in_four_rolls = (5/6)**4
prob_no_sixes_in_four_rolls

In [ ]:
# Chance of at least one six in 4 rolls of a die
1-prob_no_sixes_in_four_rolls

In [ ]:
# Chance of at least one six in n rolls of a die

rolls = np.arange(1, 51, 1)
results = Table().with_columns(
    'Rolls', rolls,
    'Chance of at least one 6', 1 - (5/6)**rolls
)
results

In [ ]:
results.scatter('Rolls')

> **The complement trick:** "at least one six" is hard to count directly (one six? two? three?) — but "no sixes at all" is easy: (5/6)⁴. Subtract from 1 and done. Watch the scatter: with 50 rolls, at least one six is nearly certain.

---
## 2. Random Sampling

From probability to data: a **sample** is a subset of a population. How you choose it is everything. (KL deck, slides 18–19.)

First, ~13,800 United flights — and some *deterministic* samples (no chance involved):

We load in a dataset of all United flights national flights from 6/1/15 to 8/9/15, their destination and how long they were delayed, in minutes.

In [ ]:
united = Table.read_table('united.csv')
united = united.with_column('Row', np.arange(united.num_rows)).move_to_start('Row')
united

Some deterministic samples:

In [ ]:
united.where('Destination', 'JFK') 

In [ ]:
united.take(np.arange(0, united.num_rows, 1000))

In [ ]:
united.take(make_array(34, 6321, 10040))

A random sample:

In [ ]:
start = np.random.choice(np.arange(1000))
systematic_sample = united.take(np.arange(start, united.num_rows, 1000))
systematic_sample.show()

> **Deterministic vs. random:** `where` and `take` with fixed positions give the same rows every time. The systematic sample above *is* random — the random start decides everything that follows.

---
## 3. Distributions: Probability vs. Empirical

A die has a **probability distribution**: exactly 1/6 per face, forever. A *sample* of rolls has an **empirical distribution**: whatever actually happened. Watch them differ — then converge.

In [ ]:
die = Table().with_column('Face', np.arange(1, 7))
die

In [ ]:
die.sample(10)

In [ ]:
die.hist()

In [ ]:
roll_bins = np.arange(0.5, 6.6, 1)

In [ ]:
die.hist(bins=roll_bins)

In [ ]:
die.sample(10).hist(bins=roll_bins)

### Try It 1: More rolls 😊

1. Simulate 1000 rolls of a die and plot the empirical distribution. What do you notice compared to 10 rolls?

2. Now 10,000 rolls. What do you notice?

In [ ]:
# Fill in — replace the ... 😅
...

In [ ]:
# Fill in — replace the ... 😅
...

<details>
<summary>💡 Possible answers — click to peek</summary>
<br>

*`die.sample(n)` rolls n times; more rolls → flatter, closer to the true 1/6-per-face distribution.*

```python
# 1.
die.sample(1000).hist(bins=roll_bins)

# 2.
die.sample(10000).hist(bins=roll_bins)
```

</details>

> **The Law of Averages** (KL deck, slide 21): as the number of draws grows, the empirical distribution settles toward the probability distribution. This is *why* simulation works.

---
## 4. Large Random Samples

The same idea for real data: the flight delays have a fixed (population) distribution — and a large enough random sample looks just like it.

In [ ]:
united 

In [ ]:
united_bins = np.arange(-20, 201, 5)
united.hist('Delay', bins = united_bins)

In [ ]:
min(united.column('Delay'))

In [ ]:
max(united.column('Delay'))

In [ ]:
np.average(united.column('Delay'))

### Try It 2: Sample the flights ✈️

1. Draw a sample of 10 flights and plot the distribution of delays. What do you observe?

2. Now a sample of 1000. What changed?

In [ ]:
# Fill in — replace the ... 😅
...

In [ ]:
# Fill in — replace the ... 😅
...

<details>
<summary>💡 Possible answers — click to peek</summary>
<br>

*Small samples are noisy and can look nothing like the population; big ones are near-copies.*

```python
# 1.
united.sample(10).hist('Delay', bins=united_bins)

# 2.
united.sample(1000).hist('Delay', bins=united_bins)
```

</details>

---
> **Today's takeaway:** three rules calculate chances, simulation checks them, and the Law of Averages guarantees that large random samples mirror their population. That guarantee is the foundation of inference — starting next lecture.

## Appendix — Quick Reference

Full `datascience` docs: [data8.org/datascience](https://data8.org/datascience/)

### Minimal template — sample and compare
```python
t.sample(n)                      # n random rows (with replacement)
t.sample(n).hist('col', bins=b)  # empirical distribution
```